### Include Library

In [13]:
# library for cap_f1
from cap_f1 import LLMClient, AtomicProcessor, ResultsRepo
from fewshot_examples import (
    YOUR_FEWSHOT_DEDUP_MESSAGES,
    YOUR_FEWSHOT_RECALL_MESSAGES,
    YOUR_FEWSHOT_PRECISION_MESSAGES,
)

# 1) build API + processor (inject few-shot examples if you want)
llm = LLMClient()
proc = AtomicProcessor(
    llm,
    fewshot_dedup=YOUR_FEWSHOT_DEDUP_MESSAGES,           # or None
    fewshot_recall=YOUR_FEWSHOT_RECALL_MESSAGES,         # or None
    fewshot_precision=YOUR_FEWSHOT_PRECISION_MESSAGES,   # or None
)

from datetime import datetime
import os

# code for no need for restarting the kernel when python file is updated
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Step 0: Load Data

In [14]:
print("Loading caption dataset...")

# number of data points testing
LIMIT = 1

# for filename
now = datetime.now()
timestamp = now.strftime("%Y-%m-%d_%H-%M")

#create folder to save the results
folder_path = f"results/{timestamp}"
os.makedirs(folder_path, exist_ok=True)

org_caption_dataset = ResultsRepo.read_json("data/one_data.json")

Loading caption dataset...


In [15]:
all_human_captions = []
for item in org_caption_dataset:
    # Filter out human captions that are mention quality issues
    human_captions = [
        hc["caption"]
        for hc in item["human_captions"]
        if hc["caption"] != "Quality issues are too severe to recognize visual content."
    ]
    all_human_captions.append(human_captions)

### Step 1: Parse Human & Model Caption into Atomic Statements

In [16]:
print("Generating atomic statements using gpt-4o...")

T_atomics, g_atomics, parsed_T = proc.generate_atomic_statement(org_caption_dataset, limit=LIMIT)

Generating atomic statements using gpt-4o...


100%|██████████| 1/1 [00:13<00:00, 13.35s/it]


In [17]:
# Save the parsing results
print("Saving intermediate results...")
ResultsRepo.save_results_json(    
    output_path=f"{folder_path}/intermediate_{timestamp}.json",
    org_dataset=org_caption_dataset, 
    T_atomics=T_atomics, 
    g_atomics=g_atomics, 
    parsed_T= parsed_T, 
    T_org=all_human_captions, 
    limit=LIMIT
)

Saving intermediate results...
Saved JSON to: results/2025-08-10_00-14/parsed_caption_2025-08-10_00-14.json


### Step 2: Match human & generated 
1. match human caption to model caption
2. create recall and precision data

In [19]:
# Read data from variable
# before calculating F1 score, match sentences between human generated and model generated
recall_precision = proc.evaluate_matching(all_human_captions, T_atomics, g_atomics)

ResultsRepo.save_results_json(
    output_path=f"{folder_path}/eval_{timestamp}.json",
    update_existing=f"{folder_path}/intermediate_{timestamp}.json",
    metadata=recall_precision, 
    limit=LIMIT
)

100%|██████████| 1/1 [00:25<00:00, 25.06s/it]

Saved JSON to: results/2025-08-10_00-14/recall_precision_2025-08-10_00-14.json


### Step 3: Calculate F1 Score

In [20]:
# get cap f1 score
evaluation = proc.calculate_cap_f1(recall_precision)
ResultsRepo.save_results_json(
    output_path=f"{folder_path}/final_{timestamp}.json",
    update_existing=f"{folder_path}/eval_{timestamp}.json",
    evaluations=evaluation, 
    limit=LIMIT
)

100%|██████████| 1/1 [00:00<00:00, 14315.03it/s]

Saved JSON to: results/2025-08-10_00-14/final_2025-08-10_00-14.json


In [22]:
print("Saving final results into csv...")
ResultsRepo.export_final_csv(
    json_path=f"{folder_path}/final_{timestamp}.json",
    csv_path=f"{folder_path}/final_{timestamp}.csv",
    # model_keys={"gpt":"gpt-4o-2024-08-06", "molmo":"Molmo-7B-O-0924", "llama":"Llama-3.2-11B-Vision-Instruct"}
)

Saving final results into csv...
CSV file saved to: results/2025-08-10_00-14/final_2025-08-10_00-14.csv
